### **Subemotions: Clustering de palabras por emoción** ###

In [1]:
"""Imports necesarios"""
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, precision_recall_curve, auc, roc_curve, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.cluster import KMeans
from sklearn.svm import LinearSVC




from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2

from collections import Counter
from gensim.models import fasttext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import json
import re
import os
import joblib


SEED = 1234

**Carga de datos**

In [2]:
"""Cargamos los datos"""
df_train = pd.read_csv("dreaddit/dreaddit-train.csv")
df_test = pd.read_csv("dreaddit/dreaddit-test.csv")

In [3]:
X_train = df_train["text"].astype(str)
Y_train = df_train["label"]

X_test = df_test["text"].astype(str)
y_test = df_test["label"]
print("Ejemplo de texto:", X_train[0])

Ejemplo de texto: He said he had not felt that way before, suggeted I go rest and so ..TRIGGER AHEAD IF YOUI'RE A HYPOCONDRIAC LIKE ME: i decide to look up "feelings of doom" in hopes of maybe getting sucked into some rabbit hole of ludicrous conspiracy, a stupid "are you psychic" test or new age b.s., something I could even laugh at down the road. No, I ended up reading that this sense of doom can be indicative of various health ailments; one of which I am prone to.. So on top of my "doom" to my gloom..I am now f'n worried about my heart. I do happen to have a physical in 48 hours.


**1.1. SUBEMOTIONS CON TF-IDF:**

In [4]:
def cluster_KM(X, emotion, n):
    X = np.array(X)
    print("Creating clusters.. ")
    
    clustering = KMeans(n_clusters=n, random_state=SEED).fit(X)
    
    centers = clustering.cluster_centers_
    labels = clustering.labels_
    
    print("Clusters creados:", len(centers))
    return clustering, labels

In [5]:
def get_texts_from_dir(cat_dir):
    texts, f_names = [], []
    for f_name in sorted(os.listdir(cat_dir)):
        f_path = os.path.join(cat_dir, f_name)
        with open(f_path, "r") as f:
            texts.append(f.read())
            f_names.append(f_name)
    print("%d files loaded from %s" % (len(texts), cat_dir))
    return texts, f_names

emo_words, emotions = get_texts_from_dir("emotions_ext/emotions_ext")
em_words = [[w for w in text.split("\n") if w.strip()] for text in emo_words]
print(emotions)
print(len(emotions),len(emo_words))
print(emo_words[0])
print(type(emo_words[0]),len(emo_words[0]))
print(type(em_words[0]),len(em_words[0]))
print(em_words[0])


10 files loaded from emotions_ext/emotions_ext
['emoLex_anger_ext.txt', 'emoLex_anticipation_ext.txt', 'emoLex_disgust_ext.txt', 'emoLex_fear_ext.txt', 'emoLex_joy_ext.txt', 'emoLex_negative_ext.txt', 'emoLex_positive_ext.txt', 'emoLex_sadness_ext.txt', 'emoLex_surprise_ext.txt', 'emoLex_trust_ext.txt']
10 10
abandoned
abandonment
abhor
abhorrent
abolish
abomination
abuse
accursed
accusation
accused
accuser
accusing
actionable
adder
adversary
adverse
adversity
advocacy
affront
aftermath
aggravated
aggravating
aggravation
aggression
aggressive
aggressor
agitated
agitation
agony
alcoholism
alienate
alienation
allegation
altercation
ambush
anarchism
anarchist
anarchy
anathema
anger
angry
anguish
animosity
animus
annihilate
annihilated
annihilation
annoy
annoyance
annoying
antagonism
antagonist
antagonistic
antichrist
antipathy
antisocial
antithesis
anxiety
argue
argument
argumentation
arguments
armament
armed
arraignment
arrogant
arson
assail
assailant
assassin
assassinate
assassination
a

In [6]:
ft = fasttext.load_facebook_vectors("cc.en.300.bin/cc.en.300.bin")

**K-means sobre palabras de EmoLex → centroides de subemocións**

Para cada emoción, vectorizamos as súas palabras con FastText e aplicamos K-means. Cada centroide é unha subemoción. Ao final temos unha matriz de todos os centroides e as súas etiquetas (`anger_0`, `anger_1`, ..., `joy_0`, ...).

In [ ]:
# Número de subemociones por emoción
N = 300

# Diccionario: palabra -> 'emocion_clister'
subemotion_labels = []
subemotion_centroids = []

for i, words in enumerate(em_words):
    
    emotion = emotions[i].replace(".txt", "").replace("emoLex_", "").replace("_ext", "")
    
    vectors = []
    
    for w in words:
        w = w.lower()
        if w:
            vectors.append(ft[w])

    if len(vectors) < N:
        print(f"[SKIP] {emotion}: solo {len(vectors)} palabras, necesita >= {N}")
        continue

    

    kmeans, labels = cluster_KM(vectors, emotion, N)
    
   
    for j, centroid in enumerate(kmeans.cluster_centers_):
        subemotion_labels.append(f"{emotion}_{j}")
        subemotion_centroids.append(centroid)

subemotion_centroids = np.array(subemotion_centroids)    
print(f"\nPalabras en subemotion_labels: {len(subemotion_labels)}")
print("Ejemplo:", subemotion_labels[:5])
print(f"Centroides en subemotion_centroids: {len(subemotion_centroids)}")


In [ ]:
norms = np.linalg.norm(subemotion_centroids, axis=1, keepdims=True)
centroids_normalized = subemotion_centroids / np.clip(norms, a_min=1e-8, a_max=None)

print(f"Centroides normalizados: {centroids_normalized.shape}")

**Vocabulario do dataset -> embedding -> subemoción máis cercana**

En lugar de recalcular o embedding de cada palabra en cada texto, extraemos primeiro o vocabulario completo do dataset (palabras únicas) e calculamos o seu embedding **unha soa vez**. Isto xera un diccionario `word_subemotion` que imos empregar para enmascarar.

In [ ]:
# Xuntamos os textos de train e test para facer o vocabulario
all_texts = pd.concat([X_train, X_test], ignore_index=True)
# print(all_texts)
vocabulary = set()
for text in all_texts:
    for w in text.lower().split():
        vocabulary.add(w)
        
print(f"Vocabulario único do dataset: {len(vocabulary)} palabras")

# Para cada palabra calculamos su embedding
print("Calculando embeddings e asignando subemociones...")
word_subemotion = {}
vocab = list(vocabulary)

# embeddings
vocab_vecs = np.array([ft[w] for w in vocab])
# normalizamos 
norms = np.linalg.norm(vocab_vecs, axis=1, keepdims=True)
vocab_vecs_norm = vocab_vecs / np.clip(norms, a_min=1e-10, a_max=None)  # Evitamos división por cero

print("Calculando similitudes e asignando subemocions...")
similarities = np.dot(vocab_vecs_norm, centroids_normalized.T)
closest_subemotion_indices = np.argmax(similarities, axis=1)
for w, idx in zip(vocab, closest_subemotion_indices):
    word_subemotion[w] = subemotion_labels[idx]
print(f"Palabras asignadas a subemociones: {len(word_subemotion)}")
print("Exemplo de asignación:", list(word_subemotion.items())[:5])

# otro ejemplo
for word in ['hate', 'love', 'stress', 'sleep', 'happy', 'worthless']:
    print(f"  '{word}' -> {word_subemotion.get(word, 'N/A')}")
    

In [ ]:
pipeline_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,3),
        #stop_words="english" #n-gramas de 1 a 3
    )), #non vou a limitar o vocabulario, algunhas palabras poden ser clave
    ("clf", LogisticRegression(max_iter=1000, 
                               random_state=SEED,
                               class_weight="balanced",
                               penalty="l2",
                               C=10)) ## clf es classifier, o modelo de regresión logística con un número máximo de iteraciones de 1000 para asegurar a converxencia
])

param_grid = {
    "tfidf__ngram_range": [(1,1), (1,2), (1,3)],
    "tfidf__stop_words": [None, "english"],
}

grid_search_tfidf = GridSearchCV(
    pipeline_tfidf,
    param_grid,
    cv=10,
    scoring="f1"  
)

grid_search_tfidf.fit(X_train, Y_train)

In [ ]:
best_model_tfidf = grid_search_tfidf.best_estimator_
print("Mellores hiperparámetros:", grid_search_tfidf.best_params_)

"""Evaluamos o modelo con os mellores hiperparámetros no conxunto de proba."""
y_pred = best_model_tfidf.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
tfidf = best_model_bose.named_steps["tfidf"]

X_bose_train = tfidf.transform(X_train_masked).toarray()
X_bose_test = tfidf.transform(X_test_masked).toarray()

np.save("bose_train.npy", X_bose_train)
np.save("bose_test.npy", X_bose_test)